In [1]:
#https://pypi.org/project/rank-eval/

from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
import bm25s
import Stemmer
import numpy as np
from ir_metric import metric
from sklearn.metrics import ndcg_score


In [2]:
passages_hashed = pd.read_parquet("../data/quati/passages.parquet")
qrels = pd.read_parquet("../data/quati/rel_queries_passages.parquet")
queries = pd.read_parquet("../data/quati/queries.parquet")


In [3]:
stemmer = Stemmer.Stemmer("portuguese")


In [4]:
corpus_tokens = bm25s.tokenize(passages_hashed["passage"], stopwords="pt", stemmer=stemmer)

Split strings:   0%|          | 0/1000000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [5]:
retriever = bm25s.BM25(corpus=passages_hashed.to_dict(orient='records'))
retriever.index(corpus_tokens)


BM25S Count Tokens:   0%|          | 0/1000000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1000000 [00:00<?, ?it/s]

In [6]:
queries.shape

(200, 2)

In [7]:
queries.columns

Index(['query_id', 'query'], dtype='str')

In [8]:
queries.iloc[0]

query_id                                                   0
query       Qual a maior característica da flora brasileira?
Name: 0, dtype: object

In [9]:
query_tokens = bm25s.tokenize(queries['query'], stemmer=stemmer)

Split strings:   0%|          | 0/200 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/200 [00:00<?, ?it/s]

In [10]:
results, scores = retriever.retrieve(query_tokens, k=10)


BM25S Retrieve:   0%|          | 0/200 [00:00<?, ?it/s]

In [11]:
results.shape

(200, 10)

In [90]:
results

array([[{'passage_id': 'clueweb22-pt0000-05-02747_0', 'passage': 'BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA - Klima Naturali™ Klima Home Biomas BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA BIOMAS E FITOFISIONOMIAS BRASILEIROS: CARACTERÍSTICAS E EVOLUÇÃO DA FAUNA E FLORA Tags Biomas Um bioma é um conjunto de tipos de vegetação que abrange grandes áreas contínuas, em escala regional, com flora e fauna similares, definida pelas condições físicas predominantes nas regiões. Esses aspectos climáticos, geográficos e litológicos (das rochas), por exemplo, fazem com que um bioma seja dotado de uma diversidade biológica singular, própria. No Brasil, os biomas existentes são (da maior extensão para a menor): a Amazônia, o cerrado, a Mata Atlântica, a Caatinga, o Pampa e o Pantanal. A seguir, conheça cada bioma do Brasil. Amazônia Extensão aproximada: 4.196.943 quilômetros quadrados', 'passage_hash': '06d1938e1a8b96cdcdbc

In [12]:
passage_ids_from_queries = np.array([[r['passage_id'] for r in res] for res in results])


In [102]:
passage_ids_from_queries[0][0]

np.str_('clueweb22-pt0000-05-02747_0')

In [13]:
tmp_res = []
for i, res in enumerate(results):
    for j, r in enumerate(res):
        tmp_res.append({'query_id': i, 'passage_id': r['passage_id'], 'pos': j})
res_df = pd.DataFrame(tmp_res)
res_df

,query_id,passage_id,pos
0,0,clueweb22-pt0000-05-02747_0,0
1,0,clueweb22-pt0000-27-04576_1,1
2,0,clueweb22-pt0000-69-05954_7,2
3,0,clueweb22-pt0001-26-13945_1,3
4,0,clueweb22-pt0001-73-11175_1,4
...,...,...,...
1995,199,clueweb22-pt0000-24-02686_2,5
1996,199,clueweb22-pt0000-97-10611_5,6
1997,199,clueweb22-pt0001-56-08475_13,7
1998,199,clueweb22-pt0001-77-13828_0,8


In [14]:
df_qrel = qrels.sort_values(by=['query_id', 'score'], ascending=[True, False])
df_qrel.set_index(['query_id', 'passage_id'], inplace=True)

In [15]:
res_df.set_index(['query_id', 'passage_id'], inplace=True)

In [31]:
metric_retrieve = res_df.join(df_qrel, on=['query_id', 'passage_id'], how='left', rsuffix='_res').fillna(0).reset_index()
metric_retrieve

,query_id,passage_id,pos,score
0,0,clueweb22-pt0000-05-02747_0,0,0.0
1,0,clueweb22-pt0000-27-04576_1,1,0.0
2,0,clueweb22-pt0000-69-05954_7,2,0.0
3,0,clueweb22-pt0001-26-13945_1,3,0.0
4,0,clueweb22-pt0001-73-11175_1,4,0.0
...,...,...,...,...
1995,199,clueweb22-pt0000-24-02686_2,5,0.0
1996,199,clueweb22-pt0000-97-10611_5,6,2.0
1997,199,clueweb22-pt0001-56-08475_13,7,0.0
1998,199,clueweb22-pt0001-77-13828_0,8,0.0


In [33]:
metric_retrieve['query_id'].value_counts()

query_id
0      10
1      10
2      10
3      10
4      10
       ..
195    10
196    10
197    10
198    10
199    10
Name: count, Length: 200, dtype: int64

In [38]:
metric_retrieve.query('query_id == 3').sort_values(by='pos')

,query_id,passage_id,pos,score
30,3,clueweb22-pt0000-53-16525_2,0,0.0
31,3,clueweb22-pt0000-01-05958_0,1,0.0
32,3,clueweb22-pt0001-05-04046_2,2,0.0
33,3,clueweb22-pt0001-57-17687_15,3,0.0
34,3,clueweb22-pt0001-03-01850_0,4,0.0
35,3,clueweb22-pt0001-34-18643_356,5,0.0
36,3,clueweb22-pt0001-17-12029_6,6,0.0
37,3,clueweb22-pt0000-48-02043_1,7,0.0
38,3,clueweb22-pt0002-01-09657_0,8,0.0
39,3,clueweb22-pt0000-23-21077_4,9,0.0


In [35]:
score = metric_retrieve.query('query_id == 1').sort_values(by='pos')['score'].to_numpy().reshape(1, -1)
score

array([[1., 2., 1., 0., 1., 1., 2., 0., 0., 0.]])

In [30]:
base = qrels.query('query_id == 1').sort_values(by='score', ascending=False).head(10)['score'].astype(float).to_numpy().reshape(1, -1)
base

array([[3., 2., 2., 2., 2., 2., 2., 2., 2., 2.]])

In [19]:
p = [str(e) for e in passage_ids_from_queries[1]]
p

['clueweb22-pt0000-05-02747_0',
 'clueweb22-pt0000-55-02859_0',
 'clueweb22-pt0000-78-09747_0',
 'clueweb22-pt0000-69-05954_7',
 'clueweb22-pt0000-87-14633_2',
 'clueweb22-pt0000-55-02859_3',
 'clueweb22-pt0001-77-01947_0',
 'clueweb22-pt0001-42-08363_12',
 'clueweb22-pt0000-55-12305_1',
 'clueweb22-pt0001-12-19001_0']

In [20]:
ar = qrels.query('query_id == 1').sort_values(by='score', ascending=False).head(10)['passage_id'].to_list()
ar

['clueweb22-pt0000-17-01771_19',
 'clueweb22-pt0001-01-18874_0',
 'clueweb22-pt0000-64-06285_35',
 'clueweb22-pt0000-41-11985_5',
 'clueweb22-pt0000-17-01771_1',
 'clueweb22-pt0001-06-19207_14',
 'clueweb22-pt0001-47-09592_2',
 'clueweb22-pt0001-77-01947_0',
 'clueweb22-pt0000-55-02859_0',
 'clueweb22-pt0001-46-05303_17']

In [21]:
result = metric.average_precision_at_k(ar, p, k=10)
print(result)
result = metric.precision_at_k(ar, p, k=10)
print(result)
result = metric.recall_at_k(ar, p, k=10)
print(result)

0.079
0.2
0.2


In [25]:
ndcg_score(base, score, k=10)

0.9423456905221581